Accompanying Code for: <br>
**Johannes Zeitler, Meinard Müller. "A Unified Perspective on CTC and SDTW using Differentiable DTW", submitted to IEEE Transactions on Audio, Speech, and Language Processing, 2025.**

Johannes Zeitler (johannes.zeitler@audiolabs-erlangen.de), 2025

### Description
Load evaluation metrics, print tables and plot bar charts

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
n_runs = 1

configs = []

configs.append({"ID": "CTC-A",
                "model_type": "mctc_we",
                "loss_type": "dDTW_mctc_we_fullWeights",
                "dDTW_weights": [1.0, 1.0, 1.0],
                "blank_penalty_weight": 1.0}) 

configs.append({"ID": "CTC-B",
                "model_type": "mctc_we",
                "loss_type": "dDTW_mctc_we_fullWeights",
                "dDTW_weights": [1.0, 1.0, 1.0],
                "blank_penalty_weight": 2.0}) 

configs.append({"ID": "CTC-C",
                "model_type": "mctc_we",
                "loss_type": "dDTW_mctc_we_fullWeights",
                "dDTW_weights": [1.0, 1.0, 1.0],
                "blank_penalty_weight": 1e20}) 

configs.append({"ID": "SDTW-D",
               "model_type": "standard_sigmoid",
               "loss_type": "dDTW",
               "dDTW_weights": [1.0, 1e20, 1.0]})

configs.append({"ID": "SDTW-E",
               "model_type": "standard_sigmoid",
               "loss_type": "dDTW",
               "dDTW_weights": [1.0, 1.0, 1.0]})

###################################################################################
configs.append({"ID": "CTC-A-W",
                "model_type": "mctc_we",
                "loss_type": "dDTW_mctc_we_fullWeights",
                "dDTW_weights": [0.1, 1.0, 1.0],
                "blank_penalty_weight": 1.0}) 

configs.append({"ID": "CTC-B-W",
                "model_type": "mctc_we",
                "loss_type": "dDTW_mctc_we_fullWeights",
                "dDTW_weights": [0.1, 1.0, 1.0],
                "blank_penalty_weight": 2.0}) 

configs.append({"ID": "CTC-C-W",
                "model_type": "mctc_we",
                "loss_type": "dDTW_mctc_we_fullWeights",
                "dDTW_weights": [0.1, 1.0, 1.0],
                "blank_penalty_weight": 1e20})

configs.append({"ID": "SDTW-D-W",
               "model_type": "standard_sigmoid",
               "loss_type": "dDTW",
               "dDTW_weights": [0.1, 1e20, 1.0]})

configs.append({"ID": "SDTW-E-W",
               "model_type": "standard_sigmoid",
               "loss_type": "dDTW",
               "dDTW_weights": [0.1, 1.0, 1.0]})

#######################################################################
configs.append({"ID": "EM",
               "model_type": "standard_sigmoid",
               "loss_type": "dDTW_EM",
               "dDTW_weights": [1.0, 1.0, 1.5]})

configs.append({"ID": "strong",
               "model_type": "standard_sigmoid",
               "loss_type": "strong"})

In [ ]:
results_dir = os.path.join("experiments", "results_filewise")
stats_dir = os.path.join("experiments", "logs")

metrics = ["precision", "recall", "f_measure", "cosine_sim", "binary_accuracy", "average_precision_score"]
run_colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

In [ ]:
fig, ax = plt.subplots(len(metrics), 1, figsize=(10,20))

all_results = []
for i_cf, config in enumerate(configs):

    conf_results = {}
    for m in metrics:
        conf_results[m] = [] 
    conf_results["Config"] = config["ID"]
    
    for i_run in range(n_runs):
        fn_results = os.path.join(results_dir, "%s_run%i.csv"%(config["ID"], i_run))
        if not os.path.isfile(fn_results):
            continue
        results_in = pd.read_csv(fn_results)
        results_mean = results_in[results_in.Filename=="FILEWISE MEAN"]

        for i_met, metric in enumerate(metrics):
            ax[i_met].scatter(i_cf, results_mean[metric].item(), color=run_colors[i_run], s = 50)

            conf_results[metric].append(results_mean[metric].item())



    av_results = {}
    av_results["Config"] = config["ID"]
    for m in metrics:
        mn = np.min(conf_results[m])
        mx = np.max(conf_results[m])
        me = np.mean(conf_results[m])
        md = np.median(conf_results[m])
        std = np.std(conf_results[m])
        var = np.var(conf_results[m])

        if "_" in m:
            sh = "".join([t[0] for t in m.split("_")[:2]])
        else:
            sh = m[:2]
        
        av_results["min_%s"%(sh)] = mn
        av_results["mean_%s"%(sh)] = me
        av_results["median_%s"%(sh)] = md
        av_results["max_%s"%(sh)] = mx
        av_results["var_%s"%(sh)] = var
        av_results["std_%s"%(sh)] = std
    
    all_results.append(av_results)

for m, a in zip(metrics, ax):
    a.grid()
    a.set_title(m)
    a.set_xticks(np.arange(len(configs)))
    a.set_xticklabels([config["ID"] for c in configs], rotation=45)


fig.tight_layout()
plt.show()

In [8]:
df_results = pd.DataFrame.from_dict(all_results)

In [ ]:
# print a few tables

print(df_results.to_string(float_format="%.3f", index=False))

print(df_results.loc[:,["Config"]+[m for m in df_results.keys() if "min" in m]].to_string(float_format="%.2f", index=False))

print(df_results.loc[:,["Config"]+[m for m in df_results.keys() if "max" in m]].to_string(float_format="%.2f", index=False))

print(df_results.loc[:,["Config"]+[m for m in df_results.keys() if "mean" in m]].to_string(float_format="%.2f", index=False))

print(df_results.loc[:,["Config"]+[m for m in df_results.keys() if "median" in m]].to_string(float_format="%.3f", index=False))

In [ ]:
# plot bar charts

metrics = ["fm", "ba", "cs"]

fig, ax = plt.subplots(len(metrics),1, figsize=(4,10))
yticklabels = []
for iRow, row in df_results.iterrows():
    for im, m in enumerate(metrics):
        ax[im].barh(iRow, row["median_%s"%(m)])
        ax[im].plot([row["min_%s"%(m)], row["median_%s"%(m)], row["max_%s"%(m)]], [iRow, iRow, iRow], color="k", marker="|")
    yticklabels.append(row.Config)
    
for ia, a in enumerate(ax):
    a.set_title("median %s (min-max)"%(metrics[ia]), fontsize=10)
    a.xaxis.grid(True)
    a.set_yticks(np.arange(len(yticklabels)))
    a.set_yticklabels(yticklabels, fontsize=8)

plt.tight_layout()
plt.show()